Multi-Head Attention：从单头到多头
1. 为什么需要多个 Head？
- 我们前面已经实现了单头注意力：输入 $X$ 经过一组线性投影，得到 $Q$、$K$、$V$，然后计算一次 Attention，得到一组注意力权重和一组聚合结果。
- 现在假设我们希望模型能同时用不同的关注方式处理同一份输入。例如阅读“小明把书借给小红”时，一个 Head 可以更关注“谁借出”，另一个 Head 可以更关注“借给谁”
（这里只是帮助理解，实际关注关系由训练学习，并非人为规定）。
于是有了多头注意力：
- 单头：一组 $Q,K,V$ → 一次 Attention → 一组结果。
- 多头：同一个 $X$ 通过不同的可学习投影，形成多组 $Q,K,V$ → 每组对应一个 Head 独立计算 Attention → 把各组结果拼接，再通过线性层融合。
- 关键：多头的目的，是得到多组独立的注意力权重。

                       输入 X（d_model 维）
                                │
                    ┌───────────┴───────────┐
                    ▼                       ▼
                线性投影 1               线性投影 2
                    │                       │
                    ▼                       ▼
            Q₁、K₁、V₁（各 m 维）   Q₂、K₂、V₂（各 n 维）
                    │                       │
                    ▼                       ▼
                Attention 1             Attention 2
                    │                       │
                    ▼                       ▼
                输出 1（m 维）          输出 2（n 维）
                    └───────────┬───────────┘
                                ▼
                          拼接（m+n 维）
                                │
                                ▼
                       输出线性投影（d_model 维）

- 单头只有一组注意力权重；双头则分别计算两组注意力权重，可能从不同角度聚合 Token 信息（但不保证两个 Head 一定学到不同内容）。
- 每头不必只有一半维度。 比如输入维度为 8，两个 Head 可以各为 8 维：拼接后 16 维，再投影回 8 维。
- 为什么常见实现要让每个 Head 更小？ 为了在增加 Head 数量时控制参数和计算量。例如输入维度为 8、两个 Head 各为 4 维：拼接后仍为 8 维。

经典 Transformer 通常让所有 Head 具有相同的维度 $d_{head}=d_{model}/h$。图中的 m ≠ n 是为了说明概念上并非一定要等分。

严格来说，也可以给不同 Head 分别设置 Q/K 的维度和 V 的维度；此图为了直观，把每头的三者画成相同维度。